# 01b — Missingness analysis (all 12 columns with gaps)

**Purpose.** Answer one question properly: *does a value being missing tell us anything about the patient's disease?* The original EDA tested this for only two columns (RF: p = 0.0055; Anti-CCP: p = 0.0559). Those two flags are used as model features, so the claim needs to be tested on **every** column with gaps, with a correction for running many tests at once.

**Reads:** `../data/raw/dataset.xlsx` (raw, nothing imputed) · **Writes:** `../results/missingness_tests.csv`

**Why this notebook exists (the findings that triggered it)**
1. The missing count of *every* column is an exact whole percentage of 12,085 rows (ESR 9%, CRP 20%, RF 11%, Anti-CCP 27%, HLA-B27 16%, ANA 31%, Anti-Ro 24%, Anti-La 25%, Anti-dsDNA 39%, Anti-Sm 43%, C3 14%, C4 17%). Real clinical data does not behave like that; it suggests gaps were inserted at fixed rates by a data generator.
2. A p-value of 0.0055 means something very different when you run 2 tests than when you run 12. If RF is the best of 12 columns, the corrected p-value is about 0.0055 × 12 = 0.066 — no longer significant at 5%.

**Plan (each step has its own cell)**
1. Missing rate per disease class.
2. χ² test of independence (missingness vs disease) for each column, with Cramér's V as the effect size and **Holm correction** for 12 tests.
3. A second, assumption-free method (permutation test) to confirm the χ² p-values.
4. Do missing values in different columns occur together (co-missingness)? Is the number of gaps per row what independent random gaps would give?
5. The "whole-percent" test, quantified.
6. Verdict, and what it means for the two `*_was_missing` features and for the paper.

**This notebook changes no data and no earlier notebook.** Its results decide whether the two missingness flags stay as features; that final decision is made by an ablation in notebook 04 (extended features with vs without the flags).

## 0. Setup and load

**What this does:** imports libraries, loads the raw Excel file, and confirms it is the expected dataset (12,085 rows, 15 columns). It then lists the columns that contain at least one missing value — there must be exactly 12 (every lab/serology column; `Age`, `Gender`, `Disease` are complete).

In [1]:
import os
import numpy as np
import pandas as pd
import scipy
from scipy.stats import chi2_contingency

print("pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)

RAW_PATH = "../data/raw/dataset.xlsx"
if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(f"{RAW_PATH} not found - see notebook 02 for how to obtain the dataset.")
df = pd.read_excel(RAW_PATH)
assert df.shape == (12085, 15), df.shape

cols = [c for c in df.columns if c != 'Disease' and df[c].isna().any()]
assert len(cols) == 12, cols
miss = df[cols].isna()
N = len(df)
y = df['Disease'].astype('category')
classes = list(y.cat.categories)
yc = y.cat.codes.to_numpy()
print("columns with missing values:", cols)

pandas 3.0.5 | numpy 2.5.2 | scipy 1.18.1
columns with missing values: ['ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']


## 1. Missing rate per disease class

**What this does:** for each of the 12 columns, the percentage of patients with a missing value *within each disease*, plus the overall rate.

**How to read it:** if missingness is unrelated to disease, every cell in a row is close to the row's overall rate. The sampling noise is about ±1–2 percentage points for the large classes and larger for Reactive Arthritis (only 516 patients). A column whose rate differs by 5+ points between diseases would deserve attention.

In [2]:
rate = (miss.groupby(df['Disease']).mean() * 100).T
rate['OVERALL'] = miss.mean() * 100
rate['max-min across classes'] = rate[classes].max(axis=1) - rate[classes].min(axis=1)
rate.round(1)

Disease,Ankylosing Spondylitis,Normal,Psoriatic Arthritis,Reactive Arthritis,Rheumatoid Arthritis,Sjögren's Syndrome,Systemic Lupus Erythematosus,OVERALL,max-min across classes
ESR,9.0,9.4,9.4,8.9,8.8,8.6,9.1,9.0,0.8
CRP,20.4,20.1,19.1,20.3,20.3,20.1,19.6,20.0,1.3
RF,10.1,10.8,12.3,15.7,10.6,11.0,10.0,11.0,5.7
Anti-CCP,25.3,26.4,26.8,30.6,28.3,27.9,25.2,27.0,5.5
HLA-B27,15.4,14.7,16.6,14.7,16.5,16.8,16.2,16.0,2.1
ANA,30.9,30.3,32.2,32.2,30.4,30.4,31.9,31.0,1.9
Anti-Ro,24.8,23.5,22.6,25.2,23.9,24.2,24.7,24.0,2.6
Anti-La,24.9,23.4,24.4,23.6,26.2,25.5,25.2,25.0,2.8
Anti-dsDNA,38.6,40.0,39.5,38.8,38.8,39.0,38.5,39.0,1.5
Anti-Sm,42.3,43.5,42.0,43.6,43.8,42.7,43.5,43.0,1.7


## 2. χ² test per column, effect size, and multiple-testing correction

**What this does:** for each column builds a 7 × 2 table (disease × missing/present) and tests whether missingness is independent of disease.

* **p-value (χ²):** the probability of seeing an association at least this strong if missingness were unrelated to disease.
* **Cramér's V:** the *size* of the association, from 0 (none) to 1 (perfect). With 12,085 patients even trivial associations can be "significant", so V matters as much as p. Rough guide: V < 0.05 negligible, 0.05–0.10 weak, > 0.10 worth a closer look.
* **Holm-adjusted p:** we run 12 tests, so by chance alone about one in twenty would show p < 0.05 for every 20 tests. The Holm correction controls the probability of *any* false alarm across all 12. **This adjusted value is the one to use.**
* **min expected count:** the χ² test is only reliable if every expected cell count is at least 5; the check is printed for each column.

In [3]:
def chi2_stat(tab):
    exp = np.outer(tab.sum(1), tab.sum(0)) / tab.sum()
    return ((tab - exp) ** 2 / exp).sum()

def holm(p):
    p = np.asarray(p); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p[idx]))
        adj[idx] = running
    return adj

K = len(classes)
rows = []
for c in cols:
    m = miss[c].to_numpy().astype(int)
    tab = np.bincount(yc * 2 + m, minlength=2 * K).reshape(K, 2)
    chi2, p, dof, exp = chi2_contingency(tab, correction=False)
    rows.append({'column': c, 'n_missing': int(tab[:, 1].sum()), 'chi2': chi2, 'dof': dof,
                 'p_value': p, 'cramers_v': np.sqrt(chi2 / (N * min(K - 1, 1))),
                 'min_expected_count': exp.min()})
tests = pd.DataFrame(rows)
tests['p_holm'] = holm(tests['p_value'])
tests['significant_after_holm(0.05)'] = tests['p_holm'] < 0.05
tests = tests.sort_values('p_value').reset_index(drop=True)
assert (tests['min_expected_count'] >= 5).all(), "chi-square assumption violated"
print("Reference: the original EDA reported p=0.0055 for RF and p=0.0559 for Anti-CCP (2 tests only).\n")
tests.round(4)

Reference: the original EDA reported p=0.0055 for RF and p=0.0559 for Anti-CCP (2 tests only).



,column,n_missing,chi2,dof,p_value,cramers_v,min_expected_count,p_holm,significant_after_holm(0.05)
0,RF,1329,18.3054,6,0.0055,0.0389,56.7451,0.0662,False
1,Anti-CCP,3263,12.2832,6,0.0559,0.0319,139.3221,0.6153,False
2,C4,2054,5.7736,6,0.4490,0.0219,87.7008,1.0000,False
3,Anti-La,3021,5.4880,6,0.4829,0.0213,128.9893,1.0000,False
4,HLA-B27,1934,5.2775,6,0.5087,0.0209,82.5771,1.0000,False
5,Anti-Ro,2900,3.6821,6,0.7196,0.0175,123.8229,1.0000,False
6,C3,1692,3.2956,6,0.7709,0.0165,72.2443,1.0000,False
7,ANA,3746,3.2760,6,0.7735,0.0165,159.9451,1.0000,False
8,Anti-Sm,5197,2.2747,6,0.8928,0.0137,221.8992,1.0000,False
9,CRP,2417,1.5633,6,0.9552,0.0114,103.2000,1.0000,False


## 3. Cross-check with a permutation test

**What this does:** a second method that makes no distributional assumptions. Shuffle the disease labels randomly 2,000 times (which destroys any real link between disease and missingness), recompute the χ² statistic each time, and report how often the shuffled statistic is at least as large as the real one.

**Why:** if the χ² p-values and the permutation p-values agree, the conclusion does not depend on the test's assumptions. Expected: they agree to within roughly ±0.02 (the permutation p-value has its own sampling noise of that order when p is moderate).

In [4]:
rng = np.random.default_rng(42)
B = 2000
perm_p = []
for c in tests['column']:
    m = miss[c].to_numpy().astype(int)
    obs = chi2_stat(np.bincount(yc * 2 + m, minlength=2 * K).reshape(K, 2).astype(float))
    cnt = 0
    for _ in range(B):
        t = np.bincount(rng.permutation(yc) * 2 + m, minlength=2 * K).reshape(K, 2).astype(float)
        cnt += chi2_stat(t) >= obs
    perm_p.append((cnt + 1) / (B + 1))
tests['p_permutation'] = perm_p
tests['abs_diff_chi2_vs_perm'] = (tests['p_value'] - tests['p_permutation']).abs()
print("max |chi2 p - permutation p| =", round(tests['abs_diff_chi2_vs_perm'].max(), 4))
tests[['column', 'p_value', 'p_permutation', 'abs_diff_chi2_vs_perm', 'p_holm']].round(4)

max |chi2 p - permutation p| = 0.016


,column,p_value,p_permutation,abs_diff_chi2_vs_perm,p_holm
0,RF,0.0055,0.0060,0.0005,0.0662
1,Anti-CCP,0.0559,0.0470,0.0090,0.6153
2,C4,0.4490,0.4473,0.0017,1.0000
3,Anti-La,0.4829,0.4758,0.0072,1.0000
4,HLA-B27,0.5087,0.5042,0.0045,1.0000
5,Anti-Ro,0.7196,0.7036,0.0160,1.0000
6,C3,0.7709,0.7811,0.0102,1.0000
7,ANA,0.7735,0.7751,0.0016,1.0000
8,Anti-Sm,0.8928,0.8816,0.0112,1.0000
9,CRP,0.9552,0.9545,0.0007,1.0000


## 4. Are gaps in different columns related to each other?

**4a. Pairwise correlation of the missing/present indicators.** If gaps were inserted independently, every pairwise correlation should be ~0, within the noise band ±3/√N ≈ ±0.027. With 66 pairs, one pair marginally outside the band happens by chance about one run in five, so a single barely-outside pair is noise; look for pairs far outside the band. A pair far outside would mean "when column A is missing, B tends to be missing too" (e.g. tests ordered as a panel).

**4b. Number of missing values per patient vs what independence predicts.** Using each column's overall rate, we compute how many patients should have 0, 1, 2, … missing values if gaps were independent, and compare with reality. Close agreement supports independent random gaps.

In [5]:
phi = miss.astype(int).corr().to_numpy().copy()
np.fill_diagonal(phi, np.nan)
band = 3 / np.sqrt(N)
print(f"largest |pairwise correlation| = {np.nanmax(np.abs(phi)):.4f}   (noise band +/-{band:.4f})")
pairs = [(cols[i], cols[j], phi[i, j]) for i in range(12) for j in range(i + 1, 12) if abs(phi[i, j]) > band]
print("pairs outside the noise band:", pairs if pairs else "none")

per_row = miss.sum(axis=1).value_counts().reindex(range(13), fill_value=0)
dist = np.array([1.0])
for pc in miss.mean():
    dist = np.convolve(dist, [1 - pc, pc])
cmp = pd.DataFrame({'observed_patients': per_row.values, 'expected_if_independent': (dist * N).round(1)},
                   index=range(13))
cmp['diff'] = cmp['observed_patients'] - cmp['expected_if_independent']
cmp

largest |pairwise correlation| = 0.0215   (noise band +/-0.0273)
pairs outside the noise band: none


,observed_patients,expected_if_independent,diff
0,452,468.7,-16.7
1,1796,1824.4,-28.4
2,3204,3147.8,56.2
3,3191,3185.2,5.8
4,2127,2106.5,20.5
5,919,959.7,-40.7
6,313,309.0,4.0
7,69,70.8,-1.8
8,13,11.5,1.5
9,1,1.3,-0.3


## 5. The "whole-percent" test, quantified

**What this does:** for each column, finds the nearest whole percentage `k` of 12,085 and checks whether the missing count equals `round(12,085 × k / 100)` exactly.

**Why this is evidence:** if a column's missing count were an arbitrary number, the chance of it landing exactly on one of the ~101 "whole-percent" values among the 12,086 possible counts is about 101 / 12,086 ≈ 0.8%. All 12 columns doing so by accident has probability about 0.008¹² ≈ 10⁻²⁵ — i.e. not an accident. (This tells us *how* the gaps were made, not *why*; the dataset's own documentation should be checked for the generation method.)

In [6]:
n_miss = miss.sum()
pct = n_miss / N * 100
k_pct = pct.round()
exact = (n_miss == (N * k_pct / 100).round())
tbl = pd.DataFrame({'n_missing': n_miss, 'pct': pct.round(3), 'nearest_whole_pct': k_pct,
                    'count_at_that_pct': (N * k_pct / 100).round(), 'exact_match': exact})
print(tbl)
chance_each = 101 / (N + 1)
print(f"\ncolumns matching exactly: {int(exact.sum())} / 12")
print(f"chance per column if counts were arbitrary: {chance_each:.4f};  all 12 by chance: {chance_each**int(exact.sum()):.1e}")

            n_missing     pct  nearest_whole_pct  count_at_that_pct  \
ESR              1088   9.003                9.0             1088.0   
CRP              2417  20.000               20.0             2417.0   
RF               1329  10.997               11.0             1329.0   
Anti-CCP         3263  27.000               27.0             3263.0   
HLA-B27          1934  16.003               16.0             1934.0   
ANA              3746  30.997               31.0             3746.0   
Anti-Ro          2900  23.997               24.0             2900.0   
Anti-La          3021  24.998               25.0             3021.0   
Anti-dsDNA       4713  38.999               39.0             4713.0   
Anti-Sm          5197  43.004               43.0             5197.0   
C3               1692  14.001               14.0             1692.0   
C4               2054  16.996               17.0             2054.0   

            exact_match  
ESR                True  
CRP                True 

## 6. Verdict

**What this does:** prints a short, mechanical summary of everything above so the conclusion cannot be argued with after the fact.

**How to interpret (decide *before* looking at the numbers):**
* If **no** column is significant after Holm correction, the association between missingness and disease is not distinguishable from chance. The two `*_was_missing` features then carry no demonstrated clinical signal; they stay in the pipeline only if the notebook-04 ablation shows a CV benefit, and the paper must not describe them as "informative missingness".
* If one or more columns **are** significant, report them with the adjusted p and Cramér's V, and consider extending the flags to those columns.
* In either case the whole-percent evidence goes into the paper's limitations section together with what the dataset documentation says about how it was generated.

In [7]:
sig = tests.loc[tests['significant_after_holm(0.05)'], 'column'].tolist()
print(f"columns tested: {len(tests)}")
print(f"significant after Holm correction (alpha 0.05): {sig if sig else 'none'}")
print(f"largest Cramer's V: {tests['cramers_v'].max():.4f}  ({tests.loc[tests['cramers_v'].idxmax(), 'column']})")
print(f"smallest raw p: {tests['p_value'].min():.4f}  ({tests.loc[tests['p_value'].idxmin(), 'column']}) -> Holm-adjusted {tests['p_holm'].min():.4f}")
for c in ('RF', 'Anti-CCP'):
    r = tests[tests['column'] == c].iloc[0]
    print(f"{c:<9} raw p={r['p_value']:.4f}  Holm p={r['p_holm']:.4f}  Cramer's V={r['cramers_v']:.4f}")
print(f"co-missingness pairs outside noise band: {len(pairs)}")
print(f"columns with whole-percent counts: {int(exact.sum())}/12")

columns tested: 12
significant after Holm correction (alpha 0.05): none
largest Cramer's V: 0.0389  (RF)
smallest raw p: 0.0055  (RF) -> Holm-adjusted 0.0662
RF        raw p=0.0055  Holm p=0.0662  Cramer's V=0.0389
Anti-CCP  raw p=0.0559  Holm p=0.6153  Cramer's V=0.0319
co-missingness pairs outside noise band: 0
columns with whole-percent counts: 12/12


## 7. Save

**What this does:** writes the test table to `../results/missingness_tests.csv` so the numbers quoted in the paper can be traced to a file.

In [8]:
os.makedirs('../results', exist_ok=True)
tests.to_csv('../results/missingness_tests.csv', index=False)
print("Saved ../results/missingness_tests.csv")

Saved ../results/missingness_tests.csv
